# What is a British racecourse?

## Purpose

Study 03 separates three concepts that the source data can blur together:

1. **racecourse identity** — the recognised venue/institutional identity;
2. **course/track identity** — a stable physical racing course or track within that venue;
3. **route/configuration/characteristic** — a lower-level or time-varying feature of a course.

The national result is consolidated from the 60 individual British racecourse evidence notebooks. The source database's `candidate_course_label` remains a source classification and is not promoted into either a governed racecourse ID or a governed physical course/track ID.

In [ ]:
from pathlib import Path
import ast
import json

import pandas as pd

# Resolve the repository root without assuming where Jupyter was launched.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'studies').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RACECOURSE_DIR = (
    PROJECT_ROOT
    / 'studies'
    / 'jurisdictions'
    / 'great_britain'
    / 'racecourses'
)

assert RACECOURSE_DIR.exists(), f'Racecourse notebook directory not found: {RACECOURSE_DIR}'

racecourse_notebooks = sorted(RACECOURSE_DIR.glob('*.ipynb'))
assert len(racecourse_notebooks) == 60, f'Expected 60 racecourse notebooks, found {len(racecourse_notebooks)}'

print('racecourse notebooks:', len(racecourse_notebooks))

## Consolidation method

The venue notebooks are the source of truth for the national consolidation. We extract their materialised DataFrame assignments rather than manually reproducing venue facts in this notebook. This keeps later venue corrections — such as Carlisle — flowing into the national result automatically.

In [ ]:
def assigned_names(source):
    """Return simple variable names assigned by a code cell."""
    tree = ast.parse(source)
    names = set()
    for node in ast.walk(tree):
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            for target in targets:
                if isinstance(target, ast.Name):
                    names.add(target.id)
    return names


def extract_dataframe(notebook_path, variable_name):
    """Execute only the cell assigning a self-contained DataFrame variable."""
    notebook = json.loads(notebook_path.read_text())
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        try:
            names = assigned_names(source)
        except SyntaxError:
            continue
        if variable_name not in names:
            continue
        namespace = {'pd': pd}
        exec(source, namespace)
        value = namespace.get(variable_name)
        if isinstance(value, pd.DataFrame):
            return value.copy()
    raise RuntimeError(f'{variable_name!r} not found as a DataFrame in {notebook_path.name}')

In [ ]:
# Consolidate source-label -> governed racecourse mappings from all venue notebooks.
source_label_mapping = pd.concat(
    [extract_dataframe(path, 'source_label_mapping') for path in racecourse_notebooks],
    ignore_index=True,
    sort=False,
)

print('source labels:', source_label_mapping['candidate_course_label'].nunique())
print('governed racecourse identities:', source_label_mapping['racecourse_identity'].nunique())

assert source_label_mapping['candidate_course_label'].nunique() == 65
assert source_label_mapping['racecourse_identity'].nunique() == 60

source_label_mapping.sort_values(['racecourse_identity', 'candidate_course_label']).head(20)

## Source labels are not racecourses

The 65 British source labels consolidate to **60 governed racecourse identities**. The many-to-one cases include venue/surface distinctions such as Kempton/Kempton (AW), Lingfield/Lingfield (AW), Newcastle/Newcastle (AW), Newmarket/Newmarket (July), and Southwell/Southwell (AW).

This is enough to reject a simple `COUNT(DISTINCT candidate_course_label)` as a count of British racecourses.

In [ ]:
# Consolidate the course/track inventory exactly as governed by the venue notebooks.
course_inventory = pd.concat(
    [extract_dataframe(path, 'course_inventory') for path in racecourse_notebooks],
    ignore_index=True,
    sort=False,
)

print('racecourses:', course_inventory['racecourse_identity'].nunique())
print('course/track inventory records:', len(course_inventory))

assert course_inventory['racecourse_identity'].nunique() == 60
assert len(course_inventory) == 90

course_inventory.sort_values(['racecourse_identity', 'course_or_track_name']).head(30)

## Inventory records are not automatically stable identities

The 90 venue inventory rows include some successive surface states or temporary configurations. Study 03 therefore separates the **inventory-record grain** from the **stable physical identity grain**.

Three already-resolved cases need collapsing:

- **Southwell:** Fibresand and Tapeta are successive surfaces of the same all-weather Flat track.
- **Newcastle:** the former Flat turf track and later Tapeta track are successive states of the same persistent Flat-track identity.
- **Windsor:** the dated 2024/25 and 2025/26 Jump layouts are temporary configurations of the Windsor turf course, not additional permanent course identities.

In [ ]:
# Start from venue-governed names, then collapse only the temporal-state cases
# established by the venue research. No other identities are merged heuristically.
stable_course_inventory = course_inventory.copy()
stable_course_inventory['stable_course_identity'] = stable_course_inventory['course_or_track_name']

resolved_collapses = {
    ('Southwell', 'All-Weather Flat Track — Fibresand'): 'All-Weather Flat Track',
    ('Southwell', 'All-Weather Flat Track — Tapeta'): 'All-Weather Flat Track',
    ('Newcastle', 'Former Flat Turf Track'): 'Flat Track',
    ('Newcastle', 'All-Weather Tapeta Track'): 'Flat Track',
    ('Windsor', 'Traditional Figure-of-Eight Turf Course'): 'Windsor Turf Course',
    ('Windsor', '2024/25 Jump Extended Left-Hand Oval'): 'Windsor Turf Course',
    ('Windsor', '2025/26 Jump Figure-of-Eight Configuration'): 'Windsor Turf Course',
}

for (racecourse, raw_identity), stable_identity in resolved_collapses.items():
    mask = (
        stable_course_inventory['racecourse_identity'].eq(racecourse)
        & stable_course_inventory['course_or_track_name'].eq(raw_identity)
    )
    stable_course_inventory.loc[mask, 'stable_course_identity'] = stable_identity

stable_identities = (
    stable_course_inventory[
        ['racecourse_identity', 'stable_course_identity']
    ]
    .drop_duplicates()
    .sort_values(['racecourse_identity', 'stable_course_identity'])
    .reset_index(drop=True)
)

identity_counts = (
    stable_identities.groupby('racecourse_identity')
    .size()
    .rename('stable_course_identities')
    .reset_index()
)

print('inventory records:', len(course_inventory))
print('stable course/track identities:', len(stable_identities))
print('racecourses with multiple stable identities:', (identity_counts['stable_course_identities'] > 1).sum())
print('
distribution:')
print(identity_counts['stable_course_identities'].value_counts().sort_index())

assert len(stable_identities) == 86
assert (identity_counts['stable_course_identities'] > 1).sum() == 20

stable_identities

## Carlisle closeout correction

The national closeout audit exposed an under-modelled venue: Carlisle had originally been represented by one conservative generic turf-course row. A second evidence pass established four separately relevant peer course identities in the venue notebook: **Flat Course, Chase Course, Inner Hurdle Course, and Outer Hurdle Course**.

The exact historical `OLD HURDLE` / `NEW HURDLE` mapping onto modern inner/outer terminology remains unresolved, but this is now a naming-history issue rather than a course-count blocker.

In [ ]:
# Derive the latest recorded surface state for each stable identity.
# Sorting valid_from ensures historical surface states do not overwrite later ones.
surface_history = stable_course_inventory.copy()
surface_history['surface_normalised'] = surface_history['surface'].astype('string').str.strip().str.lower()
surface_history['_valid_from_sort'] = pd.to_datetime(surface_history['valid_from'], errors='coerce')

latest_surface = (
    surface_history
    .sort_values(['racecourse_identity', 'stable_course_identity', '_valid_from_sort'])
    .groupby(['racecourse_identity', 'stable_course_identity'], as_index=False)
    .tail(1)
)

print('latest stable-identity surfaces:')
print(latest_surface['surface_normalised'].value_counts(dropna=False))

racecourse_surface_profile = (
    latest_surface.groupby('racecourse_identity')['surface_normalised']
    .agg(lambda s: ' + '.join(sorted(set(s.dropna().astype(str)))))
    .value_counts()
)

print('
racecourse surface profiles:')
print(racecourse_surface_profile)

## Surface is a characteristic, not the identity

The stable-identity model keeps surface separate from identity. This is necessary because Newcastle's Flat Track changed from turf to Tapeta and Southwell's all-weather track changed from Fibresand to Tapeta without requiring a new persistent track identity.

The same principle applies more broadly: surface, configuration, operational status and other mutable characteristics should be time-bounded attributes of a stable course/track identity.

In [ ]:
# Consolidate only explicitly unresolved venue questions for closeout.
unresolved_tables = []
for path in racecourse_notebooks:
    table = extract_dataframe(path, 'unresolved_questions')
    if len(table):
        table = table.copy()
        table['source_notebook'] = path.name
        unresolved_tables.append(table)

study03_unresolved = (
    pd.concat(unresolved_tables, ignore_index=True, sort=False)
    if unresolved_tables
    else pd.DataFrame()
)

print('unresolved records:', len(study03_unresolved))
if len(study03_unresolved):
    print('racecourses with unresolved records:', study03_unresolved['racecourse_identity'].nunique())
    display(study03_unresolved)

## Remaining unresolved questions

The closeout audit no longer contains an unresolved question that changes the governed national peer-course count. Remaining items concern bounded details such as exact temporal boundaries, geometry, operational use, naming or historical terminology.

They remain explicit rather than being converted into false precision.

In [ ]:
# Harvest assertion-level provenance tables from every venue notebook.
# Provenance is consolidated, not reconstructed from notebook prose.
provenance_tables = []
for path in racecourse_notebooks:
    notebook = json.loads(path.read_text())
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        try:
            names = assigned_names(source)
        except SyntaxError:
            continue
        provenance_names = {name for name in names if 'provenance' in name.lower()}
        if not provenance_names:
            continue
        namespace = {'pd': pd}
        try:
            exec(source, namespace)
        except Exception:
            continue
        for name in sorted(provenance_names):
            value = namespace.get(name)
            if isinstance(value, pd.DataFrame) and len(value):
                table = value.copy()
                table['provenance_table'] = name
                table['source_notebook'] = path.name
                provenance_tables.append(table)

study03_provenance = pd.concat(provenance_tables, ignore_index=True, sort=False)

required = ['source_authority', 'source_title', 'source_url', 'accessed_date']
for column in required:
    assert column in study03_provenance.columns, f'Missing provenance column: {column}'
    assert study03_provenance[column].fillna('').astype(str).str.strip().ne('').all(), f'Missing {column}'

study03_sources = (
    study03_provenance[required]
    .drop_duplicates()
    .sort_values(['source_authority', 'source_title', 'source_url'])
    .reset_index(drop=True)
)

print('assertion-level provenance records:', len(study03_provenance))
print('bibliographic source records:', len(study03_sources))
print('unique source URLs:', study03_sources['source_url'].nunique())
print('source notebooks:', study03_provenance['source_notebook'].nunique())

assert study03_provenance['source_notebook'].nunique() == 60

study03_sources

## Sources / provenance

Study 03 retains assertion-level provenance in the individual racecourse notebooks and consolidates it above for completeness checking. The bibliographic table is derived from that register; it is not a substitute for assertion-level evidence.

The closeout corrections to Carlisle, Ayr and Newmarket were recorded in their corresponding venue notebooks before the national result was rebuilt.

## Study 03 conclusion — What is a British racecourse?

The evidence supports a distinction between **racecourse identity** and **course/track identity**.

A British racecourse is best treated as the recognised racing venue or institutional identity at which racing takes place. That venue may contain one or several distinct physical racing courses/tracks. Beneath those stable identities there may also be named routes, temporary configurations and time-varying characteristics.

Across the 60 governed racecourse identities in Study 03, the venue notebooks now produce:

- **90 course/track inventory records**;
- **86 stable course/track identities** after resolving known temporal surface states and temporary configurations;
- **20 racecourses** with more than one stable course/track identity.

The 86 is a governed inventory of positively supported peer course/track identities at the Study 03 modelling level. It is not a claim that Britain contains exactly 86 named physical route distinctions of every possible granularity. Lower-level routes and unresolved geometry are deliberately not promoted into peer identities without sufficient evidence.

### Modelling implication

The governed model should support at least:

`racecourse -> course/track -> time-bounded characteristics`

Where useful, a lower route/configuration layer may sit beneath course/track, but Study 03 does not attempt to govern every named route nationally.

Stable course/track identity should remain separate from mutable characteristics such as surface, configuration and operational status.

### Source-data implication

`candidate_course_label` remains a source-derived classification field. It should not be treated as either a governed racecourse identifier or a governed physical course/track identifier.

### Bottom line

**A British racecourse is a venue, not necessarily a single racing course.** Where the physical racing environment matters analytically, the relevant course/track identity must be modelled separately.

In [ ]:
# Final study-level invariants: fail loudly if a venue edit changes the governed result.
assert source_label_mapping['racecourse_identity'].nunique() == 60
assert source_label_mapping['candidate_course_label'].nunique() == 65
assert len(course_inventory) == 90
assert len(stable_identities) == 86
assert (identity_counts['stable_course_identities'] > 1).sum() == 20
assert len(study03_provenance) > 0
assert study03_provenance['source_notebook'].nunique() == 60

print('Study 03 national consolidation checks passed.')